# 13 — LSTNet v2 no OD (EF01): protocolo L=2304 + purge/embargo + 5 fatias + covariáveis

Quarto notebook do protocolo v2 (após 10-baseline-ph, 11-baseline-od e 12-v2-lstnet-ph). LSTNet1D do **03** com as mudanças documentadas abaixo (detalhes no §7–§8); todo o resto espelha o 12. Janelamento/purge/val **idênticos ao 11** (splitter verbatim do §5). 2025 intocado (benchmark futuro).

## Diff exato vs 03 (`notebooks/03-lstnet-od.ipynb`)

| | 03 (v1) | 13 (v2, este notebook) |
|---|---|---|
| Janela | `L=8640 → H=288` (30 d) | **`L=2304 → H=288` (8 d, protocolo v2)** |
| Entrada do modelo | cauda `LN=2016` da janela + `tod_sin/cos` (conv `in_channels=3`) | **cauda `LN=2016` da janela (arquitetura idêntica: mesmos tamanhos de camada) + 7 canais determinísticos → conv `in_channels=3→8`** |
| Canais extras | — | **`tod_sin/cos` (como no 03) + elevação solar/90 + 4 Fourier do dia-do-ano (`f1..f4`) — futuro conhecido, sem leakage (funções verbatim de `/tmp/v2split/validate_split.py`)** |
| Strides treino/val | `8/4` (custo do `L=8640`) | **`4/4, idêntico ao 12` (a janela v2 menor permite)** |
| RevIN / AR | RevIN por janela + AR-288 | **idênticos ao 03** |
| Val | 4 fatias por data de fim, sem purge (S1 parcial: 657 janelas) | **5 fatias do 11 + purge/embargo ±H (idêntico ao 11)** |
| Seeds | 1 (42) | **5 seeds `[42, 7, 123, 2024, 999]` — mesmo treino/early-stopping do 03 por seed** |
| Régua v1 (referência) | `lstnet val 0,1380` (03) | **citada nos textos; régua v2 dos baratos vem do 11 (sazonal-naive val 0,1736)** |

Dimensões das camadas **inalteradas**: `CONV 32/k12/s6 · GRU 64 · skip GRUCell 32/p48 · head (64+32)→288 · AR 288→288 · dropout 0,1`. Só o `Conv1d.in_channels` muda (3→8: valor + 7 covariáveis). Parâmetros: 137.506 (03) → ~139.426 (+1.920 do conv).

## Protocolo v2 (travado, idêntico ao 11)

- `L=2304 → H=288` (8 d → 1 d, passo 5 min), interpolação `time` limite 24, descarte de janelas com NaN.
- Val = 5 fatias por data de fim: 19–28/abr, 20–29/jul, 15–24/set, 20–24/nov [5 d], **13–22/dez [10 d, verão]** → val 12.960 ([2880, 2880, 2880, 1440, 2880]); treino pós-purge 77.141 + purge 4.320 + NaN-desc 8.109 (do 11 executado; OD só tem micro-outages → 5 fatias cheias).
- Purge/embargo: treino exclui janelas cujo alvo `[fim−H, fim]` intersecte qualquer fatia estendida `±H` (gap mín +289 passos; v1 era −288). Splitter `purge_train`/`signed_gap_steps` **verbatim** do 11/`validate_split.py` — ver §5 (trava por `assert`).

## Execução remota (UM job por vez — 12c/23 GB estouram com concorrência)

- **Solo (máquina livre):** `.venv/bin/jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=7200 notebooks/13-v2-lstnet-od.ipynb`
- **Compartilhada:** `OMP_NUM_THREADS=4 MKL_NUM_THREADS=4 OPENBLAS_NUM_THREADS=4 .venv/bin/jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=7200 notebooks/13-v2-lstnet-od.ipynb`
- Nunca `sleep` dentro do comando remoto (o canal MCP expira); `pkill -f` com o truque `[.]` (ex.: `pkill -f 'nbconvert.*13-v2-lstnet-o[d]'`).
- Tempo estimado: **5× o treino do 03** — o 03 levou ~7 min solo (442 s, 22 épocas, 8.259 janelas/época) → **~60–90 min solo / ~2–3 h compartilhada** p/ as 5 seeds (cada seed tem ~19,3k janelas/época, ~2,3× o 03 pelo stride 8→4 + treino maior; o early-stopping define o número exato de épocas).
- Prophet/ARIMA **não** correm aqui (só LSTNet + 3 baratos de referência).

## Saídas (criadas pela execução)

`resultados/13-v2-lstnet-od/`: `metricas_val_por_seed.csv` (5 seeds, val pooled) · `metricas_val_media_dp.csv` (média±dp pooled) · `metricas_por_fatia.csv` (25 linhas: 5 fatias × 5 seeds) · `metricas_por_fatia_media_dp.csv` (5 linhas: média±dp por fatia) · `metricas_por_dia.csv` (45 dias-âncora) · `modelos/lstnet_od_s{seed}.pt` (×5) + `modelos/normalizacao.json` · `figs/` 01-eda/02-limpeza/03-stl/04-forecasts/05-mae/06-val-dias/07-curvas-treino (espelho do 03; 04 com banda média±dp das 5 seeds, 07 com as 5 curvas + média±dp). O `README.md` do experimento (formato do 03 + seção “Protocolo v2”) é escrito **após** a execução, com números reais + procedência remota (host + work dir).

Convenção: nada in-place em 00–12 · nada de `src/` · nada de 2025 neste notebook.

In [1]:
import json
import os
import random
import time
import warnings
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110})

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "dados" / "treino").exists())
CSV = ROOT / "dados/treino/ef01-mogi-das-cruzes_oxigenio-dissolvido_2024.csv"
OUT = ROOT / "resultados" / "13-v2-lstnet-od"
(OUT / "modelos").mkdir(parents=True, exist_ok=True)
(OUT / "figs").mkdir(parents=True, exist_ok=True)

# Protocolo v2 (travado, idêntico ao 11)
L, H = 2304, 288  # 8 d -> 1 d (passo 5 min)
SEASON = 288
INTERP_LIMIT = 24  # 2 h
VAL_SLICES = [("2024-04-19", "2024-04-28"), ("2024-07-20", "2024-07-29"),
              ("2024-09-15", "2024-09-24"), ("2024-11-20", "2024-11-24"),
              ("2024-12-13", "2024-12-22")]
# Arquitetura LSTNet1D do 03 (tamanhos idênticos; só in_channels 3->8)
LN, HN = 2016, 288
N_COV, N_CH = 7, 8  # tod_sin/cos + solar + f1..f4 = 7; + valor = 8
CONV_CH, CONV_K, CONV_S = 32, 12, 6
GRU_H, SKIP_H, SKIP_P = 64, 32, 48
AR_Q = 288
BATCH, LR = 256, 1e-3
MAX_EPOCHS, PATIENCE = 60, 10  # espelho do 03, por seed
TRAIN_STRIDE, VAL_STRIDE = 4, 4  # idêntico ao 12 (o 03 usava 8/4 pelo custo do L=8640)
DROPOUT = 0.1
SEEDS = [42, 7, 123, 2024, 999]  # 42 = padrão do repo; 5 reps p/ média±dp
DEVICE = torch.device("cpu")

print("ROOT:", ROOT, "| CSV existe:", CSV.exists(), "| torch:", torch.__version__)
print(f"L={L} H={H} LN={LN} HN={HN} canais={N_CH} (1 valor + {N_COV} cov) fatias={len(VAL_SLICES)} seeds={SEEDS}")
print("threads:", {k: os.environ.get(k, "<unset>") for k in
      ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS")},
      "| cpu:", os.cpu_count())
# Estimativa: 5 treinos ≈ 5× o 03 (~60–90 min solo) — UM job por vez (ver cabeçalho).
print("OUT:", OUT)

ROOT: /home/marcos/temporal-model | CSV existe: True | torch: 2.14.0+cpu
L=2304 H=288 LN=2016 HN=288 canais=8 (1 valor + 7 cov) fatias=5 seeds=[42, 7, 123, 2024, 999]
threads: {'OMP_NUM_THREADS': '<unset>', 'MKL_NUM_THREADS': '<unset>', 'OPENBLAS_NUM_THREADS': '<unset>'} | cpu: 12
OUT: /home/marcos/temporal-model/resultados/13-v2-lstnet-od


## 1. Carga


In [2]:
df = pd.read_csv(CSV, sep=";", decimal=",", encoding="windows-1252",
                 skiprows=1, parse_dates=["Data hora"], dayfirst=True, na_values=[""])
df = df.rename(columns={"Data hora": "ds", "Oxigênio Dissolvido (mg/L)": "y"}).sort_values("ds").reset_index(drop=True)
print(df.shape, df["ds"].min(), "→", df["ds"].max())
print("faltantes:", int(df['y'].isna().sum()), f"({100*df['y'].isna().mean():.1f}%)")
df.describe()

(105121, 2) 2024-01-01 00:00:00 → 2024-12-31 00:00:00
faltantes: 594 (0.6%)


,ds,y
count,105121,104527.000000
mean,2024-07-01 12:00:00,4.491787
min,2024-01-01 00:00:00,0.790000
25%,2024-04-01 06:00:00,2.810000
50%,2024-07-01 12:00:00,4.870000
75%,2024-09-30 18:00:00,6.000000
max,2024-12-31 00:00:00,7.710000
std,NaN,1.725755


## 2. EDA


In [3]:
isna = df["y"].isna().to_numpy()
gaps = np.diff(np.concatenate([[0], np.where(~isna)[0], [len(isna)]])) - 1
print(f"maior gap: {gaps.max()} passos = {gaps.max()*5/60:.1f} h | gaps > 24 passos: {(gaps > 24).sum()}")

fig, ax = plt.subplots(3, 1, figsize=(12, 9), sharex=False)
ax[0].plot(df["ds"], df["y"], lw=0.3)
ax[0].set_title("od EF01 2024 — série completa (treino)")
ax[0].set_ylabel("od")
df["y"].hist(bins=60, ax=ax[1])
ax[1].set_title("Distribuição")
df.assign(hora=df["ds"].dt.hour).boxplot(column="y", by="hora", ax=ax[2], grid=False)
ax[2].set_title("Ciclo diário")
ax[2].set_xlabel("hora")
fig.tight_layout()
fig.savefig(OUT / "figs" / "01-eda.png")
print("fig salva")

maior gap: 334 passos = 27.8 h | gaps > 24 passos: 3


fig salva


## 3. Limpeza


In [4]:
idx = pd.date_range(df["ds"].min(), df["ds"].max(), freq="5min")
s_raw = df.set_index("ds")["y"].reindex(idx)
print(f"slots na grade: {len(s_raw)} | linhas no CSV: {len(df)}")
s = s_raw.interpolate(method="time", limit=INTERP_LIMIT)
print(f"NaN após interpolação (limite {INTERP_LIMIT}): {int(s.isna().sum())}")
gi = np.where(s.isna().to_numpy())[0]
blocos = np.split(gi, np.where(np.diff(gi) > 1)[0] + 1) if len(gi) else []
for g in blocos:
    print(f"  outage {s.index[g[0]]} → {s.index[g[-1]]} ({len(g)} slots = {len(g)*5/60:.1f} h)")

amostra = slice("2024-09-09", "2024-09-16")
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(s_raw[amostra].index, s_raw[amostra].values, ".", ms=2, label="cru (com faltantes)")
ax.plot(s[amostra].index, s[amostra].values, lw=0.8, label=f"interpolado (limite {INTERP_LIMIT})")
ax.legend(); ax.set_title("Exemplo de preenchimento — semana 09–16/09")
fig.tight_layout(); fig.savefig(OUT / "figs" / "02-limpeza.png")
print("fig salva")

slots na grade: 105121 | linhas no CSV: 105121
NaN após interpolação (limite 24): 336
  outage 2024-02-02 13:45:00 → 2024-02-02 14:45:00 (13 slots = 1.1 h)
  outage 2024-03-11 10:55:00 → 2024-03-11 11:55:00 (13 slots = 1.1 h)
  outage 2024-03-25 15:30:00 → 2024-03-26 17:15:00 (310 slots = 25.8 h)
fig salva


## 4. ADF + STL (trecho limpo jul–ago)


In [5]:
trecho = s.loc["2024-07-15":"2024-08-31"].dropna()
stat, pval, *_ = adfuller(trecho.values)
print(f"ADF stat={stat:.2f} p-valor={pval:.3g} → {'estacionária' if pval < 0.05 else 'NÃO estacionária'}")

stl = STL(trecho.iloc[-4032:], period=SEASON, robust=True).fit()
fig = stl.plot()
fig.set_size_inches(12, 6)
fig.savefig(OUT / "figs" / "03-stl.png")
print("fig salva")

ADF stat=-6.61 p-valor=6.3e-09 → estacionária


fig salva


## 5. Janelamento (L=2304) + val em 5 fatias + purge/embargo ±H

Idêntico ao 11: janelas por data de **fim**, descarte com NaN pós-interp, treino = janelas válidas fora da val que **sobrevivem ao purge** (alvo `[fim−H, fim]` sem interseção com qualquer fatia estendida `±H`). Funções `purge_train`/`signed_gap_steps` **verbatim** de `/tmp/v2split/validate_split.py` (ver 11 §5). Trava se o purge falhar (gap < H+1 ou overlap > 0). Esperado OD: treino 77.141 + val 12.960 ([2880, 2880, 2880, 1440, 2880]) + purge 4.320 + NaN-desc 8.109 (do 11 executado; OD só tem micro-outages → 5 fatias cheias).

In [6]:
from numpy.lib.stride_tricks import sliding_window_view

v = s.to_numpy()
W = sliding_window_view(v, L + H)
ok = ~np.isnan(W).any(axis=1)
n_desc_nan = int((~ok).sum())
W = W[ok]
X, Y = W[:, :L], W[:, L:]
ends = s.index[L + H - 1:][ok]
ed = ends.date

# --- val por data de fim (5 fatias, incl. dez) ---
is_val = np.zeros(len(ends), dtype=bool)
per_slice = []
for a, b in VAL_SLICES:
    d0, d1 = pd.Timestamp(a).date(), pd.Timestamp(b).date()
    m = (ed >= d0) & (ed <= d1)
    is_val |= m
    per_slice.append(int(m.sum()))
    print(f"fatia {a} → {b}: {int(m.sum())} janelas válidas")


def purge_train(ends, is_val, slices):
    """Descarta treino cujo alvo [fim-H, fim] intersecte fatia estendida ±H.
    Verbatim de /tmp/v2split/validate_split.py (H global)."""
    keep = is_val.copy()
    drop = np.zeros(len(ends), dtype=bool)
    for a, b in slices:
        A = pd.Timestamp(a) - pd.Timedelta(minutes=5 * H)          # ini-H
        B = pd.Timestamp(b) + pd.Timedelta(days=1) - pd.Timedelta(minutes=5) \
            + pd.Timedelta(minutes=5 * H)                           # fim+H
        tgt0 = ends - pd.Timedelta(minutes=5 * H)
        hit = (~is_val) & (ends >= A) & (tgt0 <= B)  # alvo ∩ [A,B] ≠ ∅
        drop |= hit
    keep[~is_val & ~drop] = True  # treino sobrevivente
    return keep, drop  # keep=True → val ou treino válido


def signed_gap_steps(ends_tr, slices):
    """Distância (passos 5min) do alvo [fim-H,fim] à fatia mais próxima; <0 = overlap.
    Verbatim de /tmp/v2split/validate_split.py."""
    if not len(ends_tr):
        return None
    e = ends_tr.values.astype("datetime64[m]").astype(np.int64)  # min
    best = np.full(len(e), 10 ** 12)
    for a, b in slices:
        A = (pd.Timestamp(a).to_datetime64().astype("datetime64[m]").astype(int))
        B = ((pd.Timestamp(b) + pd.Timedelta(days=1) - pd.Timedelta(minutes=5))
             .to_datetime64().astype("datetime64[m]").astype(int))
        s0 = e - H * 5
        ov = (e >= A) & (s0 <= B)
        gap = np.where(e < A, (A - e) // 5, np.where(s0 > B, (s0 - B) // 5, -(np.minimum(e, B) - np.maximum(s0, A)) // 5 - 1))
        best = np.minimum(best, gap)
    return best


keep, drop = purge_train(ends, is_val, VAL_SLICES)
va = np.where(is_val)[0]
tr = np.where(keep & ~is_val)[0]
print(f"treino (pós-purge): {len(tr)} | val: {len(va)} | "
      f"descartadas (NaN): {n_desc_nan} | purge: {int(drop.sum())}")

# --- asserts de cobertura por fatia (OD: só micro-outages → 5 fatias cheias, idem 10/11) ---
esperado = [288 * ((pd.Timestamp(b) - pd.Timestamp(a)).days + 1) for a, b in VAL_SLICES]
assert per_slice == esperado, f"cobertura por fatia fora do esperado: {per_slice} vs {esperado}"
assert len(va) == sum(esperado) == 12960, f"val total inesperada: {len(va)}"
assert len(va) > 1000, "val pequena demais — reposicionar fatias!"
assert int(drop.sum()) > 0, "purge removeu zero janelas — lógica inativa?"

# --- trava do purge: gap mín ≥ H+1 e overlap zero ---
gaps = signed_gap_steps(ends[tr], VAL_SLICES)
print(f"gap mín alvo-treino→val: +{int(gaps.min())} passos (exigido ≥ {H + 1}); "
      f"treino c/ alvo∩val: {int((gaps < 0).sum())}")
assert int((gaps < 0).sum()) == 0, "LEAKAGE: há alvo de treino dentro da val!"
assert int(gaps.min()) >= H + 1, f"purge/embargo falhou: gap {int(gaps.min())} < {H + 1}"

# --- dias-âncora (23:55) na val: 45 = 10+10+10+5+10 ---
daily_mask = (ends.time == pd.Timestamp("23:55").time()) & is_val
daily_idx = np.where(daily_mask)[0]
por_dia_ct = [int(((ends[daily_idx].date >= pd.Timestamp(a).date()) &
                   (ends[daily_idx].date <= pd.Timestamp(b).date())).sum())
              for a, b in VAL_SLICES]
print("dias-âncora na val:", len(daily_idx), "por fatia:", por_dia_ct)
assert len(daily_idx) == 45, f"dias-âncora inesperados: {len(daily_idx)}"
assert por_dia_ct == [10, 10, 10, 5, 10], f"âncoras por fatia: {por_dia_ct}"

fatia 2024-04-19 → 2024-04-28: 2880 janelas válidas
fatia 2024-07-20 → 2024-07-29: 2880 janelas válidas
fatia 2024-09-15 → 2024-09-24: 2880 janelas válidas
fatia 2024-11-20 → 2024-11-24: 1440 janelas válidas
fatia 2024-12-13 → 2024-12-22: 2880 janelas válidas
treino (pós-purge): 77141 | val: 12960 | descartadas (NaN): 8109 | purge: 4320
gap mín alvo-treino→val: +289 passos (exigido ≥ 289); treino c/ alvo∩val: 0
dias-âncora na val: 45 por fatia: [10, 10, 10, 5, 10]


## 6. Métricas + baselines de referência


In [7]:
def mae(a, b): return float(mean_absolute_error(a.ravel(), b.ravel()))
def rmse(a, b): return float(np.sqrt(mean_squared_error(a.ravel(), b.ravel())))
def mape(a, b, eps=1e-6): return float(np.mean(np.abs((a - b) / np.maximum(np.abs(a), eps))) * 100)
def smape(a, b, eps=1e-6): return float(np.mean(2*np.abs(a-b) / (np.abs(a)+np.abs(b)+eps)) * 100)

def metricas(y_true, y_pred):
    return {"MAE": mae(y_true, y_pred), "RMSE": rmse(y_true, y_pred),
            "MAPE": mape(y_true, y_pred), "sMAPE": smape(y_true, y_pred)}

def cheap_preds(X_):
    return {
        "persistencia": np.repeat(X_[:, -1:], H, axis=1),
        "sazonal_naive_288": np.stack([X_[:, L - SEASON + h] for h in range(H)], axis=1),
        "media_movel_288": np.repeat(X_[:, -SEASON:].mean(axis=1, keepdims=True), H, axis=1),
    }

Xtr, Ytr, Xva, Yva = X[tr], Y[tr], X[va], Y[va]
print("treino rolante (régua v2 dos baratos; v1/lstnet-03 val 0,1380 p/ referência):")
print(pd.DataFrame({m: metricas(Ytr, p) for m, p in cheap_preds(Xtr).items()}).T.round(4).to_string())
print("val rolante (5 fatias):")
print(pd.DataFrame({m: metricas(Yva, p) for m, p in cheap_preds(Xva).items()}).T.round(4).to_string())

treino rolante (régua v2 dos baratos; v1/lstnet-03 val 0,1380 p/ referência):


                      MAE    RMSE    MAPE   sMAPE
persistencia       0.3812  0.5694  8.6819  8.6028
sazonal_naive_288  0.2356  0.3650  6.1279  6.0964
media_movel_288    0.3539  0.4763  8.3817  8.3034
val rolante (5 fatias):


                      MAE    RMSE    MAPE   sMAPE
persistencia       0.4280  0.6222  7.7187  7.6334
sazonal_naive_288  0.1736  0.2548  3.3647  3.3567
media_movel_288    0.3631  0.4801  6.6894  6.6394


## 7. Covariáveis determinísticas + canais do modelo (sem leakage)

Mudança (b) vs 03: além de `tod_sin/cos` (idênticos ao 03), 5 canais extras **determinísticos** — elevação solar + 4 Fourier do dia-do-ano — calculados só do timestamp (futuro conhecido, sem leakage). Funções `elevacao_solar`/`fourier_doy` **verbatim** de `/tmp/v2split/validate_split.py` (Mogi das Cruzes −23,52/−46,19, UTC−3). Solar entra normalizada por /90 (→ [−1, 1], constante determinística, sem estatística dos dados); Fourier e ToD já estão em [−1, 1]. RevIN normaliza só o canal de valor (igual ao 02); covariáveis entram cruas no conv.

Mudança (a) vs 03: o modelo consome a **cauda `LN=2016`** da janela `L=2304` (7 d dos 8 d) — mesma construção `Wln`/`Tln`/`rowln`/`monta` do 03, mesmos tamanhos de camada; só `Tln` passa de 2 → 7 canais.


In [8]:
LAT, LON, TZ = -23.52, -46.19, -3  # Mogi das Cruzes; ts locais (UTC-3, sem DST em 2024)


def elevacao_solar(ts, lat=LAT, lon=LON, tz=TZ):
    ts = pd.DatetimeIndex(ts)
    doy = ts.dayofyear.to_numpy() + (ts.hour.to_numpy() + ts.minute.to_numpy() / 60) / 24
    g = 2 * np.pi / 365 * (doy - 1 + (ts.hour.to_numpy() - 12) / 24)
    eq = 229.18 * (0.000075 + 0.001868 * np.cos(g) - 0.032077 * np.sin(g)
                   - 0.014615 * np.cos(2 * g) - 0.040849 * np.sin(2 * g))
    decl = (0.006918 - 0.399912 * np.cos(g) + 0.070257 * np.sin(g) - 0.006758 * np.cos(2 * g)
            + 0.000907 * np.sin(2 * g) - 0.002697 * np.cos(3 * g) + 0.00148 * np.sin(3 * g))
    tst = (ts.hour.to_numpy() * 60 + ts.minute.to_numpy()) + eq + 4 * lon - 60 * tz
    ha = np.radians(tst / 4 - 180)
    cosz = np.sin(np.radians(lat)) * np.sin(decl) + np.cos(np.radians(lat)) * np.cos(decl) * np.cos(ha)
    return 90 - np.degrees(np.arccos(np.clip(cosz, -1, 1)))


def fourier_doy(ts, n=366):
    d = pd.DatetimeIndex(ts).dayofyear.to_numpy()
    return (np.sin(2 * np.pi * d / n), np.cos(2 * np.pi * d / n),
            np.sin(4 * np.pi * d / n), np.cos(4 * np.pi * d / n))


# --- canais ToD idênticos ao 02 + solar + fourier (todos float32) ---
TOD_SIN = np.sin(2*np.pi*(s.index.hour.to_numpy()*60+s.index.minute.to_numpy())/1440.0).astype(np.float32)
TOD_COS = np.cos(2*np.pi*(s.index.hour.to_numpy()*60+s.index.minute.to_numpy())/1440.0).astype(np.float32)
SOLAR = (elevacao_solar(s.index) / 90.0).astype(np.float32)  # /90 determinístico → [-1, 1]
F1, F2, F3, F4 = [a.astype(np.float32) for a in fourier_doy(s.index)]
COV = np.stack([TOD_SIN, TOD_COS, SOLAR, F1, F2, F3, F4], axis=1).astype(np.float32)
assert COV.shape == (len(s), N_COV), COV.shape
print(f"sanity solar meio-dia jan: {float(SOLAR[s.index.get_loc('2024-01-15 12:00')]):+.3f} "
      f"meia-noite: {float(SOLAR[s.index.get_loc('2024-01-15 00:00')]):+.3f}")
print(f"sanity fourier 01/jan: {[round(float(v[0]), 3) for v in (F1, F2, F3, F4)]}")

# --- janelas nativas LN=2016 (mesma construção do 03; Tln 2->7 canais) ---
val5 = s.to_numpy().astype(np.float32)
Wln = sliding_window_view(val5, LN)
Tln = sliding_window_view(COV, LN, axis=0).transpose(0, 2, 1).astype(np.float32)
pos_end = s.index.get_indexer(ends - pd.Timedelta(minutes=5*H))
rowln = pos_end - LN + 1
print(f"Wln {Wln.shape} Tln {Tln.shape} (esperado (_, {LN}) e (_, {LN}, {N_COV}))")
print(f"janelas nativas válidas: {int((rowln >= 0).sum())}/{len(ends)}")
assert Wln.shape[1] == LN and Tln.shape[1:] == (LN, N_COV), (Wln.shape, Tln.shape)
assert int((rowln >= 0).sum()) == len(ends), "cauda LN fora da grade!"
assert Tln.shape[0] == Wln.shape[0] == len(s) - LN + 1
(OUT / "modelos").mkdir(parents=True, exist_ok=True)
json.dump({"mode": "revin-per-window", "LN": LN, "HN": HN, "L": L, "H": H,
           "channels": ["valor", "tod_sin", "tod_cos", "solar Elev/90", "f1_sinA", "f1_cosA", "f2_sinS", "f2_cosS"],
           "in_channels": N_CH, "solar_scale": 90.0, "lat_lon_tz": [LAT, LON, TZ],
           "val_slices": VAL_SLICES, "seeds": SEEDS},
          open(OUT / "modelos" / "normalizacao.json", "w"))
print("normalizacao.json salva:", open(OUT / "modelos" / "normalizacao.json").read()[:200], "...")


def monta(idxs):
    ii = np.asarray(idxs); r = rowln[ii]
    return Wln[r], Tln[r], Y[ii].astype(np.float32)


Xtr_v, Xtr_t, Ytr_ = monta(tr[::TRAIN_STRIDE])
Xva_v, Xva_t, Yva_ = monta(va[::VAL_STRIDE])
print(f"treino: {Xtr_v.shape} {Xtr_t.shape} (stride {TRAIN_STRIDE}) | val: {Xva_v.shape} {Xva_t.shape} (stride {VAL_STRIDE})")
assert Xtr_t.shape[1:] == (LN, N_COV) and Xva_t.shape[1:] == (LN, N_COV)

sanity solar meio-dia jan: +0.957 meia-noite: -0.500
sanity fourier 01/jan: [0.017, 1.0, 0.034, 0.999]


Wln (103106, 2016) Tln (103106, 2016, 7) (esperado (_, 2016) e (_, 2016, 7))
janelas nativas válidas: 94421/94421
normalizacao.json salva: {"mode": "revin-per-window", "LN": 2016, "HN": 288, "L": 2304, "H": 288, "channels": ["valor", "tod_sin", "tod_cos", "solar Elev/90", "f1_sinA", "f1_cosA", "f2_sinS", "f2_cosS"], "in_channels": 8, "so ...
treino: (19286, 2016) (19286, 2016, 7) (stride 4) | val: (3240, 2016) (3240, 2016, 7) (stride 4)


## 8. LSTNet — mesmo treino/early-stopping do 03, repetido nas 5 seeds

Arquitetura `LSTNet1D` **idêntica ao 03** exceto `Conv1d(3→8)`: conv → ReLU → GRU-64 + recurrent-skip (`GRUCell-32`, `p=48`) + cabeça direta `H=288` + atalho AR-288 + RevIN por janela. Hiperparâmetros espelhados (`BATCH=256`, `LR=1e-3`, `MAX_EPOCHS=60`, `PATIENCE=10`, `DROPOUT=0,1`, Adam/MSE; strides 4/4 como no 12 — o 03 usava 8/4 pelo custo do `L=8640`); por seed: fixa `random`/`numpy`/`torch`, DataLoader com `shuffle=True`, salva o melhor `val` em `modelos/lstnet_od_s{seed}.pt`. Régua v1 de referência: **03 lstnet val 0,1380**.

In [9]:
class LSTNet1D(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv1d(N_CH, CONV_CH, kernel_size=CONV_K, stride=CONV_S)
        self.gru = nn.GRU(CONV_CH, GRU_H, batch_first=True)
        self.skipcell = nn.GRUCell(CONV_CH, SKIP_H)
        self.head = nn.Linear(GRU_H + SKIP_H, HN)
        self.ar = nn.Linear(AR_Q, HN)
        self.drop = nn.Dropout(DROPOUT)
        self.gamma = nn.Parameter(torch.ones(1))
        self.beta = nn.Parameter(torch.zeros(1))
    def forward(self, xv, tod):
        mu = xv.mean(dim=1, keepdim=True); sg = xv.std(dim=1, keepdim=True).clamp_min(1e-3)
        vn = self.gamma * (xv - mu) / sg + self.beta
        f = self.drop(torch.relu(self.conv(torch.cat([vn.unsqueeze(1), tod.transpose(1, 2)], dim=1))))
        f = f.transpose(1, 2)
        _, h = self.gru(f)
        B, T, _ = f.shape
        hs = torch.zeros(B, SKIP_H, device=f.device)
        states = [hs]
        for t in range(T):
            prev = states[t - SKIP_P] if t - SKIP_P >= 0 else states[0]
            hs = self.skipcell(f[:, t, :], prev)
            states.append(hs)
        g = self.gamma.clamp_min(1e-3)
        yn = self.head(self.drop(torch.cat([h.squeeze(0), hs], dim=1)))
        ya = self.ar(vn[:, -AR_Q:])
        return (yn + ya - self.beta) / g * sg + mu


print(f"params (in_ch={N_CH}): {sum(p.numel() for p in LSTNet1D().parameters())} "
      f"(03 tinha 137.506 com in_ch=3; +{(N_CH-3)*CONV_CH*CONV_K} do conv)")

hists, bests, epochs_best, tempos = {}, {}, {}, {}
t_all = time.time()
for SEED in SEEDS:
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    model = LSTNet1D().to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    loss_fn = nn.MSELoss()
    Xtr_v_, Xtr_t_, Ytr__ = monta(tr[::TRAIN_STRIDE])
    Xva_v_, Xva_t_, Yva__ = monta(va[::VAL_STRIDE])
    tr_loader = DataLoader(TensorDataset(torch.from_numpy(Xtr_v_), torch.from_numpy(Xtr_t_), torch.from_numpy(Ytr__)),
                           batch_size=BATCH, shuffle=True)
    va_loader = DataLoader(TensorDataset(torch.from_numpy(Xva_v_), torch.from_numpy(Xva_t_), torch.from_numpy(Yva__)),
                           batch_size=256)
    best, patience, hist = float("inf"), 0, {"train": [], "val": []}
    t0 = time.time()
    for ep in range(1, MAX_EPOCHS + 1):
        model.train()
        tl = 0.0
        for xb, tb, yb in tr_loader:
            opt.zero_grad()
            loss = loss_fn(model(xb, tb), yb)
            loss.backward()
            opt.step()
            tl += float(loss.detach()) * len(xb)
        tl /= len(tr_loader.dataset)
        model.eval()
        vl = 0.0
        with torch.no_grad():
            for xb, tb, yb in va_loader:
                vl += float(loss_fn(model(xb, tb), yb)) * len(xb)
        vl /= len(va_loader.dataset)
        hist["train"].append(tl); hist["val"].append(vl)
        tag = ""
        if vl < best:
            best, patience, best_ep = vl, 0, ep
            torch.save({"state": model.state_dict(), "seed": SEED,
                        "cfg": {"ln": LN, "in_channels": N_CH, "conv": [CONV_CH, CONV_K, CONV_S],
                                "gru": GRU_H, "skip": [SKIP_H, SKIP_P], "ar": AR_Q}},
                       OUT / "modelos" / f"lstnet_od_s{SEED}.pt")
            tag = " *"
        else:
            patience += 1
        print(f"[s{SEED}] ep {ep:02d} train={tl:.4f} val={vl:.4f}{tag}", flush=True)
        if patience >= PATIENCE:
            print(f"[s{SEED}] early stopping na ep {ep} (best val={best:.4f} ep {best_ep})")
            break
    dt = time.time() - t0
    hists[SEED], bests[SEED], epochs_best[SEED], tempos[SEED] = hist, best, best_ep, dt
    print(f"[s{SEED}] treino em {dt:.0f}s | melhor val={best:.4f} (ep {best_ep})")
print(f"5 seeds em {time.time()-t_all:.0f}s")
print(pd.DataFrame({"best_val": bests, "best_ep": epochs_best,
                    "train_s": {k: round(v) for k, v in tempos.items()}}).T.round(4).to_string())

params (in_ch=8): 139426 (03 tinha 137.506 com in_ch=3; +1920 do conv)


[s42] ep 01 train=0.1264 val=0.0481 *


[s42] ep 02 train=0.0878 val=0.0472 *


[s42] ep 03 train=0.0825 val=0.0497


[s42] ep 04 train=0.0777 val=0.0390 *


...

[s999] ep 11 train=0.0447 val=0.0503


[s999] ep 12 train=0.0426 val=0.0524


[s999] ep 13 train=0.0399 val=0.0557


[s999] ep 14 train=0.0389 val=0.0514


[s999] early stopping na ep 14 (best val=0.0386 ep 4)
[s999] treino em 558s | melhor val=0.0386 (ep 4)
5 seeds em 2950s
             42        7        123       2024      999 
best_val    0.039    0.0398    0.043    0.0387    0.0386
best_ep     4.000    4.0000   10.000    5.0000    4.0000
train_s   553.000  529.0000  718.000  590.0000  558.0000


## 9. Inferência + tabelas (por seed, média±dp pooled e por fatia)

Inferência cheia na val (12.960 origens, sem stride) por seed; `metricas_val_por_seed.csv` = pooled por seed, `metricas_val_media_dp.csv` = média±dp pooled, `metricas_por_fatia.csv` = 5 fatias × 5 seeds, `metricas_por_fatia_media_dp.csv` = média±dp por fatia, `metricas_por_dia.csv` = 45 dias-âncora. Régua v1 de referência nos textos: **03 lstnet val 0,1380** (v1, L=8640, 4 fatias, S1 parcial) — comparação direta com v2 só após a execução.


In [10]:
@torch.no_grad()
def prevê_com(model_, idxs, batch=256):
    model_.eval()
    ii = np.asarray(idxs)
    outs = []
    for b in range(0, len(ii), batch):
        xb = torch.from_numpy(Wln[rowln[ii[b:b+batch]]])
        tb = torch.from_numpy(Tln[rowln[ii[b:b+batch]]])
        outs.append(model_(xb, tb).numpy())
    return np.concatenate(outs)


modelos = {}
for SEED in SEEDS:
    m = LSTNet1D().to(DEVICE)
    ckpt = torch.load(OUT / "modelos" / f"lstnet_od_s{SEED}.pt", map_location="cpu", weights_only=False)
    m.load_state_dict(ckpt["state"]); m.eval()
    modelos[SEED] = m
print("checkpoints carregados:", sorted(modelos))

t0 = time.time()
P_va = {sd: prevê_com(m, va) for sd, m in modelos.items()}  # (12960, 288) por seed
P_d = {sd: prevê_com(m, daily_idx) for sd, m in modelos.items()}  # (45, 288) por seed
print(f"inferência val+dias em {time.time()-t0:.0f}s | P_va[s42] {P_va[42].shape}")

# --- pooled por seed ---
rows_seed = []
for sd in SEEDS:
    mm = metricas(Yva, P_va[sd])
    rows_seed.append({"seed": sd, "MAE": mm["MAE"], "RMSE": mm["RMSE"],
                      "MAPE": mm["MAPE"], "sMAPE": mm["sMAPE"],
                      "best_epoch": epochs_best[sd], "train_s": round(tempos[sd]),
                      "best_val_mse": bests[sd]})
tab_seed = pd.DataFrame(rows_seed).set_index("seed").round(4)
tab_seed.to_csv(OUT / "metricas_val_por_seed.csv")
print("=== val pooled por seed (12.960 origens) ===")
print(tab_seed.to_string())

# --- média±dp pooled ---
med = tab_seed[["MAE", "RMSE", "MAPE", "sMAPE"]].mean()
dp = tab_seed[["MAE", "RMSE", "MAPE", "sMAPE"]].std(ddof=1)
tab_md = pd.DataFrame({"media": med, "dp": dp}).round(4)
tab_md.to_csv(OUT / "metricas_val_media_dp.csv")
print("=== val pooled média±dp (5 seeds) ===")
print(tab_md.to_string())
print(f"Régua v1 (03, L=8640 4 fatias): lstnet val MAE 0,1380 — v2 aqui: {med['MAE']:.4f}±{dp['MAE']:.4f}")

# --- por fatia: 5 × 5 ---
va_ends = ends[va]
rows_f = []
for a, b in VAL_SLICES:
    d0, d1 = pd.Timestamp(a).date(), pd.Timestamp(b).date()
    m_va = (va_ends.date >= d0) & (va_ends.date <= d1)
    ii_loc = np.where(m_va)[0]  # posições dentro de va
    for sd in SEEDS:
        mm = metricas(Yva[ii_loc], P_va[sd][ii_loc])
        rows_f.append({"fatia": f"{a}→{b}", "seed": sd, **mm})
tab_f = pd.DataFrame(rows_f, columns=["fatia", "seed", "MAE", "RMSE", "MAPE", "sMAPE"]).round(4)
tab_f.to_csv(OUT / "metricas_por_fatia.csv", index=False)
assert len(tab_f) == 25 and tab_f["fatia"].nunique() == 5, tab_f.shape
assert (tab_f["fatia"] == "2024-12-13→2024-12-22").any(), "fatia dez ausente!"
g = tab_f.groupby("fatia")
tab_fmd = pd.DataFrame({c: g[c].mean().round(4).astype(str) + "±" + g[c].std(ddof=1).round(4).astype(str)
                        for c in ["MAE", "RMSE", "MAPE", "sMAPE"]})
tab_fmd.to_csv(OUT / "metricas_por_fatia_media_dp.csv")
print("=== val por fatia — MAE por seed ===")
print(tab_f.pivot(index="fatia", columns="seed", values="MAE").to_string())
print("=== val por fatia — média±dp ===")
print(tab_fmd.to_string())

# --- por dia-âncora (45 dias; lstnet média±dp das 5 seeds + baratos) ---
Yd = Y[daily_idx]
cp_d = cheap_preds(X[daily_idx])
Pstack_d = np.stack([P_d[sd] for sd in SEEDS])  # (5, 45, 288)
mae_seed_dia = np.stack([[mae(Yd[k:k+1], P_d[sd][k:k+1]) for k in range(len(Yd))] for sd in SEEDS])  # (5,45)
datas = [str(ends[i].date()) for i in daily_idx]
fatias_d = []
for dt_ in ends[daily_idx].date:
    for a, b in VAL_SLICES:
        if pd.Timestamp(a).date() <= dt_ <= pd.Timestamp(b).date():
            fatias_d.append(f"{a}→{b}"); break
por_dia = pd.DataFrame(
    {m: [mae(Yd[k:k+1], cp_d[m][k:k+1]) for k in range(len(Yd))] for m in cp_d},
    index=datas)
for j, sd in enumerate(SEEDS):
    por_dia[f"lstnet_s{sd}"] = mae_seed_dia[j]
por_dia["lstnet_media"] = mae_seed_dia.mean(axis=0)
por_dia["lstnet_dp"] = mae_seed_dia.std(axis=0, ddof=1)
por_dia.insert(0, "fatia", fatias_d)
por_dia.to_csv(OUT / "metricas_por_dia.csv")
assert len(por_dia) == 45 and (por_dia["fatia"] == "2024-12-13→2024-12-22").sum() == 10
print("=== val dias-âncora (45): média das 5 seeds vs baratos ===")
print(por_dia[["fatia"] + list(cp_d) + ["lstnet_media", "lstnet_dp"]].round(4).to_string())
print(f"\nMelhor seed na val pooled: s{tab_seed['MAE'].idxmin()} = {tab_seed['MAE'].min():.4f}")

checkpoints carregados: [7, 42, 123, 999, 2024]


inferência val+dias em 33s | P_va[s42] (12960, 288)


=== val pooled por seed (12.960 origens) ===
         MAE    RMSE    MAPE   sMAPE  best_epoch  train_s  best_val_mse
seed                                                                   
42    0.1450  0.1975  2.8243  2.8146           4      553        0.0390
7     0.1462  0.1995  2.8395  2.8307           4      529        0.0398
123   0.1495  0.2076  2.9407  2.9398          10      718        0.0430
2024  0.1459  0.1967  2.7812  2.7909           5      590        0.0387
999   0.1469  0.1965  2.8133  2.8260           4      558        0.0386
=== val pooled média±dp (5 seeds) ===
        media      dp
MAE    0.1467  0.0017
RMSE   0.1996  0.0046
MAPE   2.8398  0.0603
sMAPE  2.8404  0.0577
Régua v1 (03, L=8640 4 fatias): lstnet val MAE 0,1380 — v2 aqui: 0.1467±0.0017


=== val por fatia — MAE por seed ===
seed                     7       42      123     999     2024
fatia                                                        
2024-04-19→2024-04-28  0.1111  0.1124  0.1189  0.1052  0.1015
2024-07-20→2024-07-29  0.1148  0.1171  0.1130  0.1299  0.1284
2024-09-15→2024-09-24  0.1678  0.1590  0.1621  0.1563  0.1611
2024-11-20→2024-11-24  0.1332  0.1393  0.1745  0.1451  0.1549
2024-12-13→2024-12-22  0.1977  0.1941  0.1917  0.1973  0.1881
=== val por fatia — média±dp ===
                                 MAE           RMSE           MAPE          sMAPE
fatia                                                                            
2024-04-19→2024-04-28  0.1098±0.0067  0.1485±0.0077  3.5448±0.2265  3.5792±0.2276
2024-07-20→2024-07-29  0.1206±0.0079   0.158±0.0066  1.9345±0.1236  1.9417±0.1331
2024-09-15→2024-09-24  0.1613±0.0043  0.2168±0.0076  2.5159±0.0748  2.4851±0.0635
2024-11-20→2024-11-24  0.1494±0.0161  0.1938±0.0155  2.4634±0.2768  2.4964±0.2931
2024

## 10. Figuras (espelho do 03; 04 com banda média±dp, 07 com 5 curvas + média±dp)


In [11]:
# --- 04-forecasts: 3 origens do treino (real × sazonal × lstnet média±dp 5 seeds) ---
Pstack_tr3 = {}
ks = [0, len(Xtr) // 2, -1]
fig, axes = plt.subplots(3, 1, figsize=(12, 9))
cp_tr = cheap_preds(Xtr)
for ax, k in zip(axes, ks):
    tf = pd.date_range(ends[tr[k]] - pd.Timedelta(minutes=5*(H-1)), ends[tr[k]], freq="5min")
    ax.plot(tf, Ytr[k], "k-", lw=1.5, label="real")
    ax.plot(tf, cp_tr["sazonal_naive_288"][k], "--", lw=1, label="sazonal-naive")
    Pk = np.stack([prevê_com(modelos[sd], np.array([tr[k]]))[0] for sd in SEEDS])
    ax.plot(tf, Pk.mean(axis=0), lw=1, alpha=0.9, label="lstnet média 5 seeds")
    ax.fill_between(tf, Pk.mean(axis=0) - Pk.std(axis=0, ddof=1),
                    Pk.mean(axis=0) + Pk.std(axis=0, ddof=1), alpha=0.2, label="±dp seeds")
    ax.set_title(f"origem {ends[tr[k]]} (régua v1/03 lstnet val 0,1380)")
    ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(OUT / "figs" / "04-forecasts.png")

# --- 05-mae: barras pooled (baratos + 5 seeds + média) ---
fig, ax = plt.subplots(figsize=(8, 4))
bar = pd.concat([pd.DataFrame({m: metricas(Yva, p) for m, p in cheap_preds(Xva).items()}).T["MAE"],
                 tab_seed["MAE"].rename(lambda s: f"lstnet_s{s}"),
                 pd.Series({"lstnet_media": tab_seed["MAE"].mean()})]).sort_values()
bar.plot.barh(ax=ax)
ax.set_title("MAE na val pooled (5 fatias, v2) — baratos + LSTNet por seed (menor = melhor)")
fig.tight_layout(); fig.savefig(OUT / "figs" / "05-mae.png")

# --- 06-val-dias: MAE por dia-âncora (baratos + lstnet média) ---
fig, ax = plt.subplots(figsize=(12, 3.5))
pdf = por_dia
for col, ls in [("sazonal_naive_288", "--"), ("lstnet_media", "-"), ("persistencia", ":")]:
    if col in pdf.columns:
        ax.plot(pd.to_datetime(pdf.index), pdf[col], ls, lw=1.1, label=col)
ax.fill_between(pd.to_datetime(pdf.index), pdf["lstnet_media"] - pdf["lstnet_dp"],
                pdf["lstnet_media"] + pdf["lstnet_dp"], alpha=0.2)
ax.set_title("od — MAE por dia-âncora na val (5 fatias sazonais, v2, incl. dez)")
ax.legend(fontsize=8); fig.autofmt_xdate()
fig.tight_layout(); fig.savefig(OUT / "figs" / "06-val-dias.png")

# --- 07-curvas-treino: 5 seeds (finas) + média±dp (espessa + banda) ---
fig, ax = plt.subplots(figsize=(8, 3.5))
for sd in SEEDS:
    ax.plot(hists[sd]["val"], lw=0.8, alpha=0.5, label=f"val s{sd}")
Lmax = max(len(hists[sd]["val"]) for sd in SEEDS)
arr_tr = np.full((len(SEEDS), Lmax), np.nan)
arr_va = np.full((len(SEEDS), Lmax), np.nan)
for j, sd in enumerate(SEEDS):
    arr_tr[j, :len(hists[sd]["train"])] = hists[sd]["train"]
    arr_va[j, :len(hists[sd]["val"])] = hists[sd]["val"]
ep = np.arange(1, Lmax + 1)
ax.plot(ep, np.nanmean(arr_tr, axis=0), "k-", lw=1.5, label="treino média")
ax.plot(ep, np.nanmean(arr_va, axis=0), "r-", lw=1.5, label="val média")
ax.fill_between(ep, np.nanmean(arr_va, axis=0) - np.nanstd(arr_va, axis=0, ddof=1),
                np.nanmean(arr_va, axis=0) + np.nanstd(arr_va, axis=0, ddof=1),
                color="r", alpha=0.2, label="val ±dp")
ax.set_title("LSTNet v2 — loss por época (5 seeds, média±dp)")
ax.set_xlabel("época"); ax.legend(fontsize=7)
fig.tight_layout(); fig.savefig(OUT / "figs" / "07-curvas-treino.png")
print("figs salvas")

figs salvas


## 11. Conclusões (preencher com números reais após a execução)

Réguas v2 acima (`metricas_val_por_seed.csv` = primária por seed; `metricas_val_media_dp.csv` = pooled média±dp; `metricas_por_fatia_media_dp.csv` mostra cada fatia, incl. dez). Régua v1 de referência: **03 lstnet val 0,1380** (L=8640, 4 fatias sem purge, S1 parcial, stride 8/4) — a comparação v1×v2 embute mudança de protocolo (L, purge, 5ª fatia, strides) + canais extras, não só o modelo. Checkpoint por seed em `modelos/` para o benchmark/NNLS futuro.

### Protocolo v2 (resumo p/ o README do experimento)

- Janelas `L=2304 → H=288` (8 d → 1 d, 5 min), interp `time` limite 24, descarte com NaN; val 5 fatias por data de fim (19–28/abr, 20–29/jul, 15–24/set, 20–24/nov [5 d], **13–22/dez [10 d, verão]**); purge/embargo ±H (gap mín +289; trava por `assert`).
- Modelo = LSTNet1D do 03 com 3 mudanças: (a) cauda `LN=2016` da janela `L=2304` (mesmos tamanhos de camada); (b) conv `in_channels 3→8` (valor + tod_sin/cos + solar/90 + f1..f4, RevIN e AR-288 iguais ao 03); (c) strides 8/4→4/4 (idêntico ao 12).
- Treino = hiperparâmetros/early-stopping do 03 por seed (`BATCH=256`, `LR=1e-3`, `MAX 60/PAT 10`, strides 4/4, Adam/MSE), seeds `[42, 7, 123, 2024, 999]`; reporte por seed + média±dp pooled e por fatia.
- OD só tem micro-outages (336 slots NaN pós-interp) → 5 fatias cheias, asserts de cobertura idênticos aos do 10/11.

### Procedência da execução (preencher no commit da execução)

- Host remoto: `temporal-remote` 192.168.1.6 · work dir: `/home/marcos/temporal-model` · data: `2026-09-16` (21:57–22:47 −03:00) · threads: `solo, sem cap (12c livres)`
- Pós-execução: escrever `resultados/13-v2-lstnet-od/README.md` (formato do 03 + seção “Protocolo v2”), indexar em `resultados/README.md` + `notebooks/README.md` + README §7 — com números reais. Não commitar `modelos/*.pt` (vão ao Release via `scripts/baixar_modelos.sh`).